In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, time

In [3]:
# Load the datasets from the provided CSV files.
nifty_spot_data = pd.read_csv('Nifty_spot_data_min_2024.csv')
call_option_data = pd.read_csv('Call-Option-Data-1min.csv')
put_option_data = pd.read_csv('Put-Option-Data-1min.csv')


In [4]:
# Display the first few rows and information about each DataFrame

print("Nifty Spot Data Head:")
print(nifty_spot_data.head())
print("\nNifty Spot Data Info:")
nifty_spot_data.info()




Nifty Spot Data Head:
          timestamp     Open     High      Low    Close      expiry
0  01-01-2024 09:15  21727.8  21737.3  21701.8  21710.4  04-01-2024
1  01-01-2024 09:16  21711.5  21720.0  21695.1  21695.3  04-01-2024
2  01-01-2024 09:17  21697.7  21711.8  21694.8  21709.6  04-01-2024
3  01-01-2024 09:18  21709.1  21712.5  21698.4  21701.6  04-01-2024
4  01-01-2024 09:19  21704.3  21708.0  21693.6  21693.8  04-01-2024

Nifty Spot Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92460 entries, 0 to 92459
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   timestamp  92460 non-null  object 
 1   Open       92460 non-null  float64
 2   High       92460 non-null  float64
 3   Low        92460 non-null  float64
 4   Close      92460 non-null  float64
 5   expiry     92460 non-null  object 
dtypes: float64(4), object(2)
memory usage: 4.2+ MB


In [5]:
print("\nCall Option Data Head:")
print(call_option_data.head())
print("\nCall Option Data Info:")
call_option_data.info()




Call Option Data Head:
             timestamp OptionType  Strike      Expiry    Open    High     Low  \
0  2024-11-04 09:15:00         CE   23700  07-11-2024  601.85  601.85  569.75   
1  2024-11-04 09:15:00         CE   25300  07-11-2024    2.25    2.95    1.85   
2  2024-11-04 09:15:00         CE   25400  07-11-2024    2.00    2.00    1.60   
3  2024-11-04 09:15:00         CE   24250  07-11-2024  228.00  228.00  179.10   
4  2024-11-04 09:15:00         CE   26600  07-11-2024    1.00    1.50    1.00   

    Close  
0  569.75  
1    1.95  
2    1.75  
3  184.90  
4    1.30  

Call Option Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950216 entries, 0 to 950215
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   timestamp   950216 non-null  object 
 1   OptionType  950216 non-null  object 
 2   Strike      950216 non-null  int64  
 3   Expiry      950216 non-null  object 
 4   Open        950216 non-nul

In [6]:
print("\nPut Option Data Head:")
print(put_option_data.head())
print("\nPut Option Data Info:")
put_option_data.info()


Put Option Data Head:
             timestamp OptionType  Strike      Expiry    Open    High     Low  \
0  2024-11-04 09:15:00         PE   23000  07-11-2024    6.00    6.95    4.00   
1  2024-11-04 09:15:00         PE   23050  07-11-2024    6.40    6.95    5.00   
2  2024-11-04 09:15:00         PE   22600  07-11-2024    2.45    2.45    1.60   
3  2024-11-04 09:15:00         PE   23850  07-11-2024   50.45   82.95   50.45   
4  2024-11-04 09:15:00         PE   24450  07-11-2024  293.05  333.00  293.05   

    Close  
0    5.25  
1    6.30  
2    1.85  
3   81.55  
4  331.70  

Put Option Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 917720 entries, 0 to 917719
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   timestamp   917720 non-null  object 
 1   OptionType  917720 non-null  object 
 2   Strike      917720 non-null  int64  
 3   Expiry      917720 non-null  object 
 4   Open        917720 non-null 

In [7]:
# --- 2. Data Preprocessing and Cleaning ---

# convert the 'timestamp' columns to datetime
nifty_spot_data['timestamp'] = pd.to_datetime(nifty_spot_data['timestamp'], format='%d-%m-%Y %H:%M')
call_option_data['timestamp'] = pd.to_datetime(call_option_data['timestamp'])
put_option_data['timestamp'] = pd.to_datetime(put_option_data['timestamp'])

# set timestamp as index
nifty_spot_data.set_index('timestamp', inplace=True)
call_option_data.set_index('timestamp', inplace=True)
put_option_data.set_index('timestamp', inplace=True)

# convert the 'expiry' / 'Expiry' columns to datetime with correct format
nifty_spot_data['expiry'] = pd.to_datetime(nifty_spot_data['expiry'], format='%d-%m-%Y')
call_option_data['Expiry'] = pd.to_datetime(call_option_data['Expiry'], format='%d-%m-%Y')
put_option_data['Expiry'] = pd.to_datetime(put_option_data['Expiry'], format='%d-%m-%Y')


In [8]:
# --- 2. Backtesting Function ---


def backtest_strategy(nifty_data, ce_data, pe_data, start_date, end_date):
   
   
    tradelog = []
 
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    
    # filter the Nifty data to include only the dates within our backtesting period.
    nifty_data = nifty_data[(nifty_data.index.date >= start_date.date()) & (nifty_data.index.date <= end_date.date())]
    
    # loop through each unique trading day in the filtered Nifty data.
    # .normalize() sets the time to 00:00:00, so we get one entry per day.
    for day in nifty_data.index.normalize().unique():
        day_str = day.strftime('%Y-%m-%d')
        
        # define the entry time (1:00 PM) and exit time (3:00 PM) for the strategy.
        entry_time = pd.to_datetime(day_str + ' 13:00:00')
        exit_time = pd.to_datetime(day_str + ' 15:00:00')
        
        # ff there's no Nifty data at our entry time (e.g., a holiday), skip to the next day.
        if entry_time not in nifty_data.index:
            continue
            
        # get the Nifty spot price at entry time to determine the ATM strike.
        nifty_spot_at_entry = nifty_data.loc[entry_time]['Close']
        
        atm_strike = round(nifty_spot_at_entry / 100) * 100
        
        # define the strike prices for our short strangle (ATM+100 for CE, ATM-100 for PE).
        ce_strike = atm_strike + 100
        pe_strike = atm_strike - 100
        
        # find the weekly expiry date 
        expiry_date = nifty_data.loc[entry_time]['expiry']
        
        # filter the call and put option data for the specific day, strikes, and expiry date.
        day_ce_data = ce_data[(ce_data.index.date == day.date()) & (ce_data['Strike'] == ce_strike) & (ce_data['Expiry'] == expiry_date)]
        day_pe_data = pe_data[(pe_data.index.date == day.date()) & (pe_data['Strike'] == pe_strike) & (pe_data['Expiry'] == expiry_date)]
        
        # skip this day when there no data 
        if day_ce_data.empty or day_pe_data.empty:
            continue
            
        # skip this day when the entry time not avaible 
        if entry_time not in day_ce_data.index or entry_time not in day_pe_data.index:
            continue
        
        # get closing price at 1:00 PM
        ce_entry_price = day_ce_data.loc[entry_time]['Close']
        pe_entry_price = day_pe_data.loc[entry_time]['Close']

        # calculate the 30% stop-loss price for each leg.
        ce_sl = ce_entry_price * 1.30
        pe_sl = pe_entry_price * 1.30
        
        #  track if a stop-loss  hit.
        ce_sl_hit = False
        pe_sl_hit = False
        
        # set the exit price to the closing price at 3:00 PM.
        ce_exit_price = day_ce_data.loc[exit_time]['Close'] if exit_time in day_ce_data.index else day_ce_data.iloc[-1]['Close']
        pe_exit_price = day_pe_data.loc[exit_time]['Close'] if exit_time in day_pe_data.index else day_pe_data.iloc[-1]['Close']
        
        # set the default exit time and reason.
        ce_exit_time = exit_time
        pe_exit_time = exit_time
        ce_exit_type = 'Exit @ 3pm'
        pe_exit_type = 'Exit @ 3pm'

        # loop minute-by-minute from entry to exit to check for stop-loss triggers.
        for time in pd.date_range(start=entry_time, end=exit_time, freq='1min'):
            # check the CALL leg's stop-loss.
            if time in day_ce_data.index and not ce_sl_hit:
                if day_ce_data.loc[time]['High'] >= ce_sl:
                    ce_sl_hit = True
                    ce_exit_price = ce_sl # Exit at the stop-loss price.
                    ce_exit_time = time
                    ce_exit_type = 'stoploss hit'
                    # As per the rule, move the other leg's stop-loss to its cost (entry price).
                    pe_sl = pe_entry_price
            
            # check the PUT leg's stop-loss.
            if time in day_pe_data.index and not pe_sl_hit:
                if day_pe_data.loc[time]['High'] >= pe_sl:
                    pe_sl_hit = True
                    pe_exit_price = pe_sl # Exit at the stop-loss price.
                    pe_exit_time = time
                    pe_exit_type = 'stoploss hit'
                    # If the call SL wasn't hit, move its SL to cost.
                    if not ce_sl_hit:
                        ce_sl = ce_entry_price
                        
        # calculate the Profit and Loss (PnL) for each option.
        # for a short position, PnL = (Entry Price - Exit Price) * Quantity
        ce_pnl = (ce_entry_price - ce_exit_price) * 25 # Assuming a lot size of 25.
        pe_pnl = (pe_entry_price - pe_exit_price) * 25
        
        # append the details of the call  and put option trade 
        tradelog.append({
            'Key': entry_time,
            'ExitTime': ce_exit_time,
            'Symbol': f'NIFTY{expiry_date.strftime("%d%b%y").upper()}{ce_strike}CE',
            'EntryPrice': ce_entry_price,
            'ExitPrice': ce_exit_price,
            'Quantity': -25, # Negative quantity for a short position.
            'PositionStatus': 'Closed',
            'Pnl': ce_pnl,
            'ExitType': ce_exit_type
        })
        
   
        tradelog.append({
            'Key': entry_time,
            'ExitTime': pe_exit_time,
            'Symbol': f'NIFTY{expiry_date.strftime("%d%b%y").upper()}{pe_strike}PE',
            'EntryPrice': pe_entry_price,
            'ExitPrice': pe_exit_price,
            'Quantity': -25,
            'PositionStatus': 'Closed',
            'Pnl': pe_pnl,
            'ExitType': pe_exit_type
        })

    # Convert the list of trades into a pandas DataFrame and return it.
    return pd.DataFrame(tradelog)


In [9]:
# Call our backtesting function with the loaded data and the specified date range.
tradelog_df = backtest_strategy(nifty_spot_data, call_option_data, put_option_data, '2024-11-04', '2024-12-31')



In [10]:
# Save the resulting tradelog DataFrame to a CSV file.

tradelog_df.to_csv('tradelog.csv', index=False)


In [11]:
print("\n--- Backtest Tradlog (Final) ---")
print(tradelog_df.head())


--- Backtest Tradlog (Final) ---
                  Key            ExitTime               Symbol  EntryPrice  \
0 2024-11-04 13:00:00 2024-11-04 14:41:00  NIFTY07NOV2424000CE      144.00   
1 2024-11-04 13:00:00 2024-11-04 15:00:00  NIFTY07NOV2423800PE      189.00   
2 2024-11-05 13:00:00 2024-11-05 13:29:00  NIFTY07NOV2424000CE      151.90   
3 2024-11-05 13:00:00 2024-11-05 15:00:00  NIFTY07NOV2423800PE      127.10   
4 2024-11-06 13:00:00 2024-11-06 14:18:00  NIFTY07NOV2424500CE       63.75   

   ExitPrice  Quantity PositionStatus       Pnl      ExitType  
0    187.200       -25         Closed -1080.000  stoploss hit  
1    125.400       -25         Closed  1590.000    Exit @ 3pm  
2    197.470       -25         Closed -1139.250  stoploss hit  
3     49.800       -25         Closed  1932.500    Exit @ 3pm  
4     82.875       -25         Closed  -478.125  stoploss hit  
